# Picotron All-in-One Training, Alignment, & PEFT Pipeline (Hugging Face Datasets)
This notebook verifies SFT, LoRA/DoRA, DPO, GRPO, and PPO utilizing datasets loaded directly from the Hugging Face Hub.

## Step 1: Environment Setup

In [ ]:
# Set base directory and clone fresh repository
%cd /kaggle/working
!rm -rf picotron
!pip uninstall -y picotron
!git clone https://github.com/Syntropy-AI-Labs/picotron.git

# Install in editable mode
%cd picotron
!pip install -e .

## Step 2: Write Training Configuration

In [ ]:
%%writefile config_all_in_one.yaml
model:
  vocab_size: 50257                      # Matching standard GPT2 vocab size
  hidden_size: 64
  num_hidden_layers: 2
  num_attention_heads: 2
  num_key_value_heads: 1
  intermediate_size: 128
  max_position_embeddings: 128
  norm_type: "rms"
  activation_type: "silu"
  rms_norm_eps: 0.000005
  bias: false

parallel:
  dp_size: 1
  zero_stage: 0

data:
  dataset_path: "data/all_in_one_train.bin"
  validation_dataset_path: "data/all_in_one_val.bin"
  sequence_length: 128
  micro_batch_size: 2
  num_workers: 1

train:
  learning_rate: 0.001
  min_learning_rate: 0.0001
  weight_decay: 0.01
  max_steps: 10
  warmup_steps: 2
  grad_accum_steps: 1
  seed: 42
  compile: false
  use_cuda_graphs: false
  mixed_precision: "bf16"
  save_checkpoint: false
  checkpoint_dir: "checkpoints/all_in_one"

## Step 3: Run Full Pipeline Execution (Dora, SFT, DPO, GRPO, PPO)
We load:
1. **wikitext** from Hugging Face for Supervised Fine-Tuning.
2. **Anthropic/hh-rlhf** from Hugging Face for DPO/GRPO Preference Tuning.

In [ ]:
# Force clear stale imports cache from Jupyter kernel memory
import sys
for mod in list(sys.modules.keys()):
    if "picotron" in mod:
        del sys.modules[mod]

import torch
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

from picotron.config import load_config_from_yaml
from picotron.models.llama import LLaMAModel
from picotron.peft.peft_model import PeftModel
from picotron.peft.sft_trainer import SFTTrainer
from picotron.rlhf.trainer import PreferenceTrainer
from picotron.rlhf.dataset import PreferenceDataset
from picotron.rlhf.ppo import ActorCritic, RewardModel, PPORolloutBuffer, PPOTrainer

cfg = load_config_from_yaml("config_all_in_one.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing training stages on: {device.upper()}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# -----------------------------------------------------------------
# 1. MODEL INITIALIZATION
# -----------------------------------------------------------------
print("\n--- Stage 1: Model Initialization ---")
base_model = LLaMAModel(cfg.model).to(device)
print("Model built successfully.")

# -----------------------------------------------------------------
# 2. ADAPTER INJECTION (LoRA/DoRA)
# -----------------------------------------------------------------
print("\n--- Stage 2: Adapter Injection (DoRA) ---")
peft_model = PeftModel(base_model, target_modules=["q_proj", "v_proj"], r=4, use_dora=True)
print("Injected target adapters.")

# -----------------------------------------------------------------
# 3. SUPERVISED FINE-TUNING (SFT) WITH HF DATASET
# -----------------------------------------------------------------
print("\n--- Stage 3: Supervised Fine-Tuning (SFT) using HF wikitext ---")
hf_sft_raw = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:20]")

# Tokenize
sft_texts = [row["text"] for row in hf_sft_raw if len(row["text"].strip()) > 10]
encodings = tokenizer(sft_texts, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

x_sft = encodings["input_ids"].to(device)
y_sft = x_sft.clone()
# Mask out first half of sequence prompt tokens
y_sft[:, :64] = -100

sft_dataset = TensorDataset(x_sft, y_sft)
sft_dataloader = DataLoader(sft_dataset, batch_size=2)

sft_trainer = SFTTrainer(config=cfg, model=peft_model, train_dataloader=sft_dataloader)
sft_loss = sft_trainer.train_step(x_sft, y_sft)
print(f"SFT single-step response-masked loss: {sft_loss:.4f}")

# Merge / Unmerge test
peft_model.merge_adapters()
peft_model.unmerge_adapters()

# -----------------------------------------------------------------
# 4. PREFERENCE ALIGNMENT WITH HF DATASET (Anthropic/hh-rlhf)
# -----------------------------------------------------------------
print("\n--- Stage 4: Preference Alignment using HF Anthropic/hh-rlhf ---")
hf_pref_raw = load_dataset("Anthropic/hh-rlhf", split="train[:20]")

formatted_pref_data = []
for row in hf_pref_raw:
    # Standard formatting helper
    chosen = row["chosen"]
    rejected = row["rejected"]
    
    # Split prompt and response completions
    prompt_end = chosen.find("\n\nAssistant:")
    if prompt_end != -1:
        prompt = chosen[:prompt_end]
        chosen_resp = chosen[prompt_end:]
        rejected_resp = rejected[prompt_end:]
        
        formatted_pref_data.append({
            "prompt": tokenizer.encode(prompt)[:64],
            "chosen": tokenizer.encode(chosen_resp)[:64],
            "rejected": tokenizer.encode(rejected_resp)[:64]
        })

pref_dataset = PreferenceDataset(formatted_pref_data, max_length=128)
pref_dataloader = DataLoader(pref_dataset, batch_size=2)
batch = next(iter(pref_dataloader))

# Reference Model
ref_model = LLaMAModel(cfg.model).to(device)

# A. Run DPO
print("Running DPO Trainer...")
dpo_trainer = PreferenceTrainer(
    config=cfg,
    model=base_model,
    ref_model=ref_model,
    train_dataloader=pref_dataloader,
    mode="dpo"
)
dpo_loss = dpo_trainer.train_step(batch)
print(f"DPO step preference loss: {dpo_loss:.4f}")

# B. Run GRPO
print("Running GRPO Trainer...")
grpo_trainer = PreferenceTrainer(
    config=cfg,
    model=base_model,
    ref_model=ref_model,
    train_dataloader=pref_dataloader,
    mode="grpo"
)
grpo_loss = grpo_trainer.train_step(batch)
print(f"GRPO step policy loss: {grpo_loss:.4f}")

# -----------------------------------------------------------------
# 5. REINFORCEMENT LEARNING WITH PPO
# -----------------------------------------------------------------
print("\n--- Stage 5: Reinforcement Learning (PPO) ---")
policy_actor_critic = ActorCritic(base_model).to(device)
reward_estimator = RewardModel(ref_model).to(device)
ppo_optimizer = torch.optim.AdamW(policy_actor_critic.parameters(), lr=0.0001)

ppo_trainer = PPOTrainer(
    actor_critic=policy_actor_critic,
    ref_model=ref_model,
    optimizer=ppo_optimizer
)

# Insert experiences into Rollout Buffer
buffer = PPORolloutBuffer()
q = torch.randint(0, cfg.model.vocab_size, (2, 64), device=device)
r = torch.randint(0, cfg.model.vocab_size, (2, 64), device=device)
logp = torch.randn((2, 64), device=device)
val = torch.randn((2, 64), device=device)
rew = torch.randn((2,), device=device)
mask = torch.ones((2, 64), device=device)
buffer.insert(q, r, logp, val, rew, mask)

ppo_loss = ppo_trainer.train_step_ppo(buffer.get_batches())
print(f"PPO trainer update step loss: {ppo_loss:.4f}")

print("\n--- ALL PIPELINE STAGES COMPLETED SUCCESSFULLY ---")